In [1]:
import pandas as pd
import numpy as np
import ast
import re
from collections import Counter
import json

import load_data as ld
import candidates as ds
import priority_queue as pq
import subgroups as s
import refinements as rf
import subgroup_selection as ss
import search as bs
import preprocessing as pp
import candidates as ca
import quality_measure as qm

In [2]:
# UNCOMMENT If you run for the first time without the CSV ready -> gets firstbatch_2

# pp.transform_jsons()

In [3]:
df = pd.read_csv('algebrakit_firstbatch_2.csv')
collection_attributes = ['skills', 'stepTypes', 'event_types', 'event_evaluations', 'positive_skills', 'negative_skills']


for colatt in collection_attributes:
    df[colatt] = df[colatt].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

df_unfiltered = df


In [4]:
# if a student has not tried one exercise, remove student session from data.
# --> later: remove that interaction. If all other failed are False, then finished = True
print(len(df))
invalid_sessions = []

for i in range(len(df)):
    
    if len( df.iloc[i]['event_types'] ) == 0:
        invalid_sessions.append( df.iloc[i]['sessionID'])


df = df [~df['sessionID'].isin(invalid_sessions)] #omit all information on sessions that have at least one empty event flow (unserious students?)

df = df.reset_index().drop(['index'], axis=1)
print(len(df))
df.head(2)

6637
5441


,exerciseID,sessionID,success,creationTimestamp,skills,stepTypes,finished,marksEarned,marksTotal,questionID,...,result_progress,marksEarned_interaction,marksTotal_interaction,finished_interaction,maxAttempts,usedAttempts,failed,marksPenalty,hintsRequested,mathErrors
0,9f4a675e-c2b3-4e11-91cc-4445951bab60,040b59bb-b5da-4ef0-ba5b-3e23f163c7de,True,1741946748,[FractionArithmeticMultiplicationProper],[CALCULATE],True,4.0,4.0,Q1,...,1.0,1.0,1.0,True,2.0,0.0,False,0.0,0,0
1,9f4a675e-c2b3-4e11-91cc-4445951bab60,040b59bb-b5da-4ef0-ba5b-3e23f163c7de,True,1741946748,[FractionArithmeticMultiplicationProper],[CALCULATE],True,4.0,4.0,Q2,...,1.0,1.0,1.0,True,2.0,0.0,False,0.0,0,0


In [5]:
list( df.iloc[:5]['failed'])

[False, False, False, False, False]

In [6]:
print(len(df))

df_clean_list = []

for sesh in list( df_unfiltered['sessionID'].unique()):
    df_sesh = df_unfiltered[ df_unfiltered['sessionID'] == sesh ].copy()
    
    invalid_indices = df_sesh.index[df_sesh['event_types'].apply(lambda x: x == [])].tolist()
    
    if not(invalid_indices): #no action needed
        df_clean_list.append(df_sesh)
        continue

    df_sesh = df_sesh.drop(invalid_indices)
        
    if True in list(df_sesh['failed']):
        df_clean_list.append(df_sesh)
        continue #finished does not change
    else:
        df_sesh.loc[:, 'finished'] = True
        df_clean_list.append(df_sesh)
    
    

df_clean = pd.concat(df_clean_list, ignore_index=True)   

5441


In [7]:
df_unfiltered[['event_types', 'failed', 'finished', 'exerciseID', 'sessionID', 'interactionID']].head(10)

,event_types,failed,finished,exerciseID,sessionID,interactionID
0,[EVALUATE],False,False,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,Ex8wE
1,"[EVALUATE, EVALUATE, HINT, EVALUATE, GIVEUP]",True,False,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,gBslt
2,[],NaN,False,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,OPqyZ
3,[],NaN,False,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,IakQk
4,"[EVALUATE, HINT, EVALUATE, EVALUATE]",False,False,9f4a675e-c2b3-4e11-91cc-4445951bab60,03bba4c5-afea-42c0-87bb-f99e5f2d2bfb,Ex8wE
5,[],NaN,False,9f4a675e-c2b3-4e11-91cc-4445951bab60,03bba4c5-afea-42c0-87bb-f99e5f2d2bfb,gBslt
6,[],NaN,False,9f4a675e-c2b3-4e11-91cc-4445951bab60,03bba4c5-afea-42c0-87bb-f99e5f2d2bfb,OPqyZ
7,[],NaN,False,9f4a675e-c2b3-4e11-91cc-4445951bab60,03bba4c5-afea-42c0-87bb-f99e5f2d2bfb,IakQk
8,[EVALUATE],False,True,9f4a675e-c2b3-4e11-91cc-4445951bab60,040b59bb-b5da-4ef0-ba5b-3e23f163c7de,Ex8wE
9,[EVALUATE],False,True,9f4a675e-c2b3-4e11-91cc-4445951bab60,040b59bb-b5da-4ef0-ba5b-3e23f163c7de,gBslt


In [8]:
df_clean[['event_types', 'failed', 'finished', 'exerciseID', 'sessionID', 'interactionID']].head(10)

,event_types,failed,finished,exerciseID,sessionID,interactionID
0,[EVALUATE],False,False,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,Ex8wE
1,"[EVALUATE, EVALUATE, HINT, EVALUATE, GIVEUP]",True,False,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,gBslt
2,"[EVALUATE, HINT, EVALUATE, EVALUATE]",False,True,9f4a675e-c2b3-4e11-91cc-4445951bab60,03bba4c5-afea-42c0-87bb-f99e5f2d2bfb,Ex8wE
3,[EVALUATE],False,True,9f4a675e-c2b3-4e11-91cc-4445951bab60,040b59bb-b5da-4ef0-ba5b-3e23f163c7de,Ex8wE
4,[EVALUATE],False,True,9f4a675e-c2b3-4e11-91cc-4445951bab60,040b59bb-b5da-4ef0-ba5b-3e23f163c7de,gBslt
5,[EVALUATE],False,True,9f4a675e-c2b3-4e11-91cc-4445951bab60,040b59bb-b5da-4ef0-ba5b-3e23f163c7de,OPqyZ
6,[EVALUATE],False,True,9f4a675e-c2b3-4e11-91cc-4445951bab60,040b59bb-b5da-4ef0-ba5b-3e23f163c7de,IakQk
7,[EVALUATE],False,False,9f4a675e-c2b3-4e11-91cc-4445951bab60,05a619b7-8441-4b47-b891-f9d0f7471f43,Ex8wE
8,"[EVALUATE, EVALUATE]",False,False,9f4a675e-c2b3-4e11-91cc-4445951bab60,05a619b7-8441-4b47-b891-f9d0f7471f43,gBslt
9,[EVALUATE],False,False,9f4a675e-c2b3-4e11-91cc-4445951bab60,05a619b7-8441-4b47-b891-f9d0f7471f43,OPqyZ


In [9]:
print( df['finished'].value_counts())

finished
False    3638
True     1803
Name: count, dtype: int64


In [10]:
print( len( list(df['sessionID'].unique())))

1338


In [11]:
print( df_unfiltered['finished'].value_counts())

finished
False    4834
True     1803
Name: count, dtype: int64


In [12]:
print( len( list(df_unfiltered['sessionID'].unique())))

1622


In [13]:
print( df_clean['finished'].value_counts())

finished
False    3958
True     2062
Name: count, dtype: int64


In [14]:
print( len( list(df_clean['sessionID'].unique())))

1622


# Data distribution in the Use Case
Our data can be split into two categories: the **exercise attributes** that describe the contents of an exercise and the **student attributes** that describe the workflow and performance of the students that made the exercise. The hierarchical level that each attribute is at, is denoted behind the attribute name.

**Exercise Attributes:**
* skills (eid)
* stepTypes (eid)
* marksTotal (eid)
* itemType (qid)
* interactionType (iid)
* solution (iid)
* content (iid)
* interaction marksTotal (iid)
* maxAttempts (iid)

**Student Attributes:**
* success (sid)
* finished (sid)
* marksEarned (sid)
* eventTypes (iid)
* eventEvaluations (iid)
* positive skills (iid)
* negative skills (iid)
* result status (iid)
* result progress (iid)
* interaction marksEarned (iid)
* interaction finished (iid)
* usedAttempts (iid)
* failed (iid)
* marksPenalty (iid)
* hintsRequested (iid)
* mathErrors (iid)

## Attribute Types and Descriptive and Target Space.
That we have this many attributes, does not mean that we select all attributes. We therefore select only a part of it. These attributes can be composed of six categories: numerical (float), ordinal (int), categorical, boolean, collection, sequence, and subsequence. The last three are list-type attributes that simulate the aggregation of a hierarchical level that has no need to be expanded into seperate rows in the data. 

**Target Attribute**
* finished (boolean)

The below mentioned are our descriptors:

**Numerical (float)**
* marksPenalty

**Ordinal (int)**
* marksTotal
* interaction marksTotal
* usedAttempts
* marksPenalty
* hintsRequested
* mathErrors

**Categorical**
* itemType
* interactionType

**Boolean**
* nvt

**Collection (list, no order)**
* skills
* positive skills
* negative skills

**Sequence (list, order immediate)**
* eventTypes
* eventEvaluations

**Subsequence (list, order with pauses)**
* stepTypes (also possible as collection or sequence)

In [15]:
df.columns

Index(['exerciseID', 'sessionID', 'success', 'creationTimestamp', 'skills',
       'stepTypes', 'finished', 'marksEarned', 'marksTotal', 'questionID',
       'questionType', 'interactionID', 'itemType', 'interactionType',
       'solution', 'content', 'event_types', 'event_evaluations',
       'positive_skills', 'negative_skills', 'result_status',
       'result_progress', 'marksEarned_interaction', 'marksTotal_interaction',
       'finished_interaction', 'maxAttempts', 'usedAttempts', 'failed',
       'marksPenalty', 'hintsRequested', 'mathErrors'],
      dtype='object')

In [16]:
descriptors = [ 'itemType', 'interactionType', #categorical
               'skills', 'positive_skills', 'negative_skills', #collection 
               'event_types', 'event_evaluations',  #sequence
               'stepTypes', #subsequence
               'marksTotal', 'marksTotal_interaction', 'usedAttempts', 'marksPenalty', 'hintsRequested', 'mathErrors'] #ordinal


targets = ['finished'] # all to do with performance

ids = ['exerciseID', 'sessionID', 'questionID', 'interactionID']

In [17]:
# +- 12 mins
refinement_params = {'numerical_bins': 3, 'collection_limit': 3}
beam_params = {'k': 10, 'd': 3, 'w': 20}
constraint_params = {'minimum_size': 25}
model_params = {'model': 'F_beta', 'beta': 1, 'weighted': True, 'entropy level': 'exercise'}

result_set = bs.beam_search(data=df, descriptors=descriptors, targets=targets, ids=ids, 
                            refinement_params=refinement_params, beam_params=beam_params,
                            constraint_params=constraint_params, model_params=model_params)

start on level 1
[{'interactionType': 'MULTISTEP'}, {'interactionType': 'FILL_IN_THE_BLANKS'}, {'skills': ('FractionArithmeticMultiplicationProper',)}, {'skills': ('AlgebraProductOfPowerFunctions',)}, {'skills': ('NumberArithmeticAdditionInteger',)}, {'skills': ('NumberArithmeticMultiplicationDecimal',)}, {'skills': ('NumberArithmeticPower',)}, {'skills': ('FractionArithmeticMultiplicationMixed',)}, {'skills': ('NumberArithmeticMultiplicationInteger',)}, {'skills': ('FractionArithmeticSimplifyProper',)}, {'skills': ('EquationsBalanceMethodArrangeTerms',)}, {'skills': ('EquationsOperationsAdd',)}, {'skills': ('AlgebraAddingLikeTerms',)}, {'skills': ('EquationsBalanceMethodDivideCoefficient',)}, {'skills': ('EquationsOperationsMultiply',)}, {'skills': ('FractionArithmeticEquivalentConcept',)}, {'skills': ('FactorSingleBrackets',)}, {'skills': ('NumberArithmeticAdditionDecimal',)}, {'skills': ('FractionArithmeticAddProperCommonDenominator',)}, {'skills': ('ArithmeticOrderDivision',)}, {'s

In [18]:
result_set.summarize_queue()

This queue has subgroups of average size 2.2 in a queue of length 10 and an average quality of 0.07036825297235183.


In [19]:
for sub in result_set._queue:
    print(sub)
    print(f" avg. recall = {np.mean(sub._metrics['recall'])}")
    print(f" avg. precision = {np.mean(sub._metrics['precision'])}")
    print(f" avg. F_beta = {np.mean(sub._metrics['F_beta'])}")
    print(f" avg. goodness_weights = {np.mean(sub._metrics['goodness_weights'])}")
    print(f" avg. fail rate = {np.mean(sub._metrics['fail rates'])}")
    print(f" entropy = {sub._metrics['entropy']}")
    print(f" sessions = {sub._metrics['session_counts']}")
    print(56 * "=========")

This subgroup of size 3 has description {'skills': ('FractionArithmeticSimplifyProper', 'AlgebraAddingLikeTerms')} and a quality score of 0.08562561530197298.
 avg. recall = 0.8042813455657493
 avg. precision = 0.7603740949931885
 avg. F_beta = 0.7503434986872556
 avg. goodness_weights = 0.8051391223155928
 avg. fail rate = 0.7603740949931885
 entropy = 0.14061089270397698
 sessions = [84, 68, 61]
This subgroup of size 3 has description {'skills': ('FractionArithmeticSimplifyProper', 'AlgebraAddingLikeTerms'), 'positive_skills': ('AlgebraAddingLikeTerms',)} and a quality score of 0.07704335755215755.
 avg. recall = 0.6527206173234651
 avg. precision = 0.7294068504594821
 avg. F_beta = 0.6684793928690972
 avg. goodness_weights = 0.8051391223155928
 avg. fail rate = 0.7294068504594821
 entropy = 0.14061089270397698
 sessions = [70, 57, 60]
This subgroup of size 1 has description {'positive_skills': ('FractionArithmeticAddProperCommonDenominator',)} and a quality score of 0.07384275247819

In [20]:
df_rates = pd.DataFrame.from_dict(qm.finished_rates(df_unfiltered)[0]).T.reset_index()
df_rates.iloc[-1] = ['total', sum(df_rates[False]), sum(df_rates[True])]
df_rates['fail_rate'] = [ round( ( df_rates.iloc[i][False] ) / ( df_rates.iloc[i][True] + df_rates.iloc[i][False] ) * 100, 2) for i in range(len(df_rates))]
df_rates['pass_rate'] = [ round( ( df_rates.iloc[i][True] ) / ( df_rates.iloc[i][True] + df_rates.iloc[i][False] ) * 100, 2) for i in range(len(df_rates))]
df_rates

,index,False,True,fail_rate,pass_rate
0,086f5a82-c080-4d05-9431-03e992a0d7b3,118,5,95.93,4.07
1,0beaa764-f558-4124-8d4d-b224ebdb2d4a,94,16,85.45,14.55
2,264d9ab3-57ba-4861-855b-fcc2a465c5db,114,10,91.94,8.06
3,315bc927-cdd9-4d2a-8b3c-68f830f74e70,48,43,52.75,47.25
4,486d7a7a-f282-48f4-8b74-c01a1c23a971,69,28,71.13,28.87
5,58b0759d-48ef-49f3-bb5f-7d999f35b12b,57,27,67.86,32.14
6,9aec3c1a-0100-40dd-a146-bdee6ff72fb3,127,40,76.05,23.95
7,9f4a675e-c2b3-4e11-91cc-4445951bab60,96,31,75.59,24.41
8,a04508aa-09fa-4782-bfc3-effc3789fd47,35,51,40.70,59.30
9,ba15ab94-5da4-414e-a01d-107ea7c34d67,113,16,87.60,12.40


In [21]:
df_rates = pd.DataFrame.from_dict(qm.finished_rates(df)[0]).T.reset_index()
df_rates.iloc[-1] = ['total', sum(df_rates[False]), sum(df_rates[True])]
df_rates['fail_rate'] = [ round( ( df_rates.iloc[i][False] ) / ( df_rates.iloc[i][True] + df_rates.iloc[i][False] ) * 100, 2) for i in range(len(df_rates))]
df_rates['pass_rate'] = [ round( ( df_rates.iloc[i][True] ) / ( df_rates.iloc[i][True] + df_rates.iloc[i][False] ) * 100, 2) for i in range(len(df_rates))]
df_rates

,index,False,True,fail_rate,pass_rate
0,086f5a82-c080-4d05-9431-03e992a0d7b3,79,5,94.05,5.95
1,0beaa764-f558-4124-8d4d-b224ebdb2d4a,80,16,83.33,16.67
2,264d9ab3-57ba-4861-855b-fcc2a465c5db,99,10,90.83,9.17
3,315bc927-cdd9-4d2a-8b3c-68f830f74e70,38,43,46.91,53.09
4,486d7a7a-f282-48f4-8b74-c01a1c23a971,46,28,62.16,37.84
5,58b0759d-48ef-49f3-bb5f-7d999f35b12b,41,27,60.29,39.71
6,9aec3c1a-0100-40dd-a146-bdee6ff72fb3,83,40,67.48,32.52
7,9f4a675e-c2b3-4e11-91cc-4445951bab60,58,31,65.17,34.83
8,a04508aa-09fa-4782-bfc3-effc3789fd47,28,51,35.44,64.56
9,ba15ab94-5da4-414e-a01d-107ea7c34d67,109,16,87.20,12.80


In [22]:
df_rates = pd.DataFrame.from_dict(qm.finished_rates(df_clean)[0]).T.reset_index()
df_rates.iloc[-1] = ['total', sum(df_rates[False]), sum(df_rates[True])]
df_rates['fail_rate'] = [ round( ( df_rates.iloc[i][False] ) / ( df_rates.iloc[i][True] + df_rates.iloc[i][False] ) * 100, 2) for i in range(len(df_rates))]
df_rates['pass_rate'] = [ round( ( df_rates.iloc[i][True] ) / ( df_rates.iloc[i][True] + df_rates.iloc[i][False] ) * 100, 2) for i in range(len(df_rates))]
df_rates

,index,False,True,fail_rate,pass_rate
0,086f5a82-c080-4d05-9431-03e992a0d7b3,96,27,78.05,21.95
1,0beaa764-f558-4124-8d4d-b224ebdb2d4a,88,22,80.00,20.00
2,264d9ab3-57ba-4861-855b-fcc2a465c5db,108,16,87.10,12.90
3,315bc927-cdd9-4d2a-8b3c-68f830f74e70,44,47,48.35,51.65
4,486d7a7a-f282-48f4-8b74-c01a1c23a971,58,39,59.79,40.21
5,58b0759d-48ef-49f3-bb5f-7d999f35b12b,47,37,55.95,44.05
6,9aec3c1a-0100-40dd-a146-bdee6ff72fb3,102,65,61.08,38.92
7,9f4a675e-c2b3-4e11-91cc-4445951bab60,75,52,59.06,40.94
8,a04508aa-09fa-4782-bfc3-effc3789fd47,34,52,39.53,60.47
9,ba15ab94-5da4-414e-a01d-107ea7c34d67,113,16,87.60,12.40
